# Step 5 — Predictive Analyst (the fine-tune target ⭐)

Predict the hardest investor questions from **peer lags + negative sentiment + retrieved
precedent** (RAG over the wiki). Each question carries evidence (`fact_id`s + transcript URLs).
This agent is the target of the **critical fine-tuning** step (`docs/finetuning.md`):
in `LLM_BACKEND=vllm` the (fine-tuned) model phrases/ranks; offline we use grounded templates.

In [1]:
import sys
from pathlib import Path
def _root():
    p = Path.cwd()
    for d in (p, *p.parents):
        if (d / "requirements.txt").exists():
            return d
    return p
ROOT = _root()
sys.path.insert(0, str(ROOT / "src"))
from ir_copilot.config import settings
print("backends -> qdrant:", settings.qdrant_mode, "| embeddings:", settings.embedding_backend,
      "| sentiment:", settings.sentiment_backend, "| llm:", settings.llm_backend)

backends -> qdrant: memory | embeddings: hash | sentiment: lexicon | llm: mock


In [2]:
from ir_copilot.facts import build_fact_store
from ir_copilot.embeddings import get_embedder
from ir_copilot.vectorstore import WikiStore, WikiChunk
from ir_copilot import corpus
from ir_copilot.agents.sentiment import analyze_sentiment
from ir_copilot.agents.competitor import compare
from ir_copilot.agents.predictive import predict_questions
from ir_copilot.llm import get_chat

store = build_fact_store(settings.ticker, settings.period, use_mock=settings.use_mock_data)
peers = {p: build_fact_store(p, settings.period, use_mock=settings.use_mock_data) for p in settings.peers}
wiki = WikiStore(get_embedder()); wiki.ensure_collection(recreate=True)
wiki.upsert([WikiChunk(chunk_id=str(i), **c) for i, c in enumerate(corpus.TRANSCRIPT_CHUNKS)])
items = [it for it in (corpus.NEWS_HEADLINES + corpus.SOCIAL_POSTS) if it["ticker"] == settings.ticker]

snap = analyze_sentiment(settings.ticker, items)
pc = compare(store, peers)
questions = predict_questions(store, snap, pc, wiki=wiki, chat=get_chat("analyst"))

print(f"Predicted {len(questions)} questions (ranked by difficulty):\n")
for i, q in enumerate(questions, 1):
    print(f"{i}. [{q.difficulty:.2f}] {q.text}")
    print(f"     why: {q.rationale}")
    print(f"     evidence: {q.evidence}")
assert questions, "expected predicted questions"

Predicted 2 questions (ranked by difficulty):

1. [1.00] Investors are concerned about competitive pressure / market share. How are you addressing it?
     why: Negative sentiment theme (score -1.0). Precedent: "Analyst: How should we think about customer concentration risk as a few large cloud custom..."
     evidence: ['https://example.com/news/3', 'https://example.com/post/2', 'https://example.com/nvda/fy25q3#qa1']
2. [1.00] Investors are concerned about regulatory / export restrictions. How are you addressing it?
     why: Negative sentiment theme (score -1.0). Precedent: "Analyst: How should we think about customer concentration risk as a few large cloud custom..."
     evidence: ['https://example.com/news/5', 'https://example.com/news/5', 'https://example.com/news/5', 'https://example.com/post/3', 'https://example.com/nvda/fy25q3#qa1']


/Users/v843010/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


**Next (Step 6):** draft the script/deck/Q&A and enforce grounding.